In [ ]:
"""
Korean Text Word2Vec Training
- tsa.txt 파일을 읽어 '/' 기준으로 문장 분리
- KoNLPy Okt 형태소 분석기로 토큰 추출 (명사/동사/형용사)
- Gensim Word2Vec으로 학습 후 유사 단어 출력
"""

import re
from konlpy.tag import Okt
from gensim.models import Word2Vec

# ── 1. 파일 읽기 & 문장 분리 ────────────────────────────────────
FILE_PATH = "tsa.txt"  # 실행 위치에 맞게 경로 수정

with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw = f.read()

# '/' 기준으로 문장 분리 후 공백/빈 문장 제거
sentences_raw = [s.strip() for s in raw.split("/") if s.strip()]
print(f"[1] 총 문장 수: {len(sentences_raw)}")
for i, s in enumerate(sentences_raw, 1):
    print(f"  [{i}] {s[:40]}{'...' if len(s) > 40 else ''}")

# ── 2. 형태소 분석 & 토큰 추출 ──────────────────────────────────
okt = Okt()
TARGET_POS = {"Noun", "Verb", "Adjective"}  # 추출할 품사

tokenized = []
print("\n[2] 형태소 분석 결과")
for i, sent in enumerate(sentences_raw, 1):
    sent_clean = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", sent)
    tokens = [
        word for word, pos in okt.pos(sent_clean, norm=True, stem=True)
        if pos in TARGET_POS and len(word) > 1
    ]
    tokenized.append(tokens)
    print(f"  [{i}] {tokens}")

# ── 3. Word2Vec 학습 ─────────────────────────────────────────────
model = Word2Vec(
    sentences=tokenized,
    vector_size=100,   # 임베딩 차원
    window=5,          # 컨텍스트 윈도우 크기
    min_count=1,       # 최소 등장 횟수
    workers=4,
    epochs=100,
    sg=1,              # 1=Skip-gram, 0=CBOW
)

model.save("word2vec_tsa.model")
print("\n[3] 모델 저장 완료: word2vec_tsa.model")
print(f"    학습 어휘 수: {len(model.wv)}")
print(f"    어휘 목록: {list(model.wv.index_to_key)}")

# ── 4. 유사 단어 조회 예시 ───────────────────────────────────────
test_words = ["선거", "트럼프", "이란", "공천", "선박"]
print("\n[4] 유사 단어 Top 20")
for word in test_words:
    if word in model.wv:
        similar = model.wv.most_similar(word, topn=20)
        print(f"  '{word}' → {similar}")
    else:
        print(f"  '{word}' → 어휘에 없음")

In [ ]:
"""
Korean Text Word2Vec Training + n-gram 기반 문장 생성 (동적 길이 + 다양성 + OOV 처리)
- tsa_cleaned.txt 파일을 읽어 '/' 기준으로 문장 분리
- KoNLPy Okt 형태소 분석기로 토큰 추출
- Gensim Word2Vec으로 학습
- Bigram/Trigram + Word2Vec 유사도 + Temperature 샘플링으로 문장 생성
- OOV 처리: 형태소 분리 → 편집거리 대체 순으로 fallback
"""

import re, math, random
from collections import defaultdict
from konlpy.tag import Okt
from gensim.models import Word2Vec

# ── 1. 파일 읽기 & 문장 분리 ────────────────────────────────────
FILE_PATH = "tsa.txt"

with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw = f.read()

sentences_raw = [s.strip() for s in raw.split("/") if s.strip()]
print(f"[1] 총 문장 수: {len(sentences_raw)}")

# ── 2. 형태소 분석 & 토큰 추출 ──────────────────────────────────
okt = Okt()
TARGET_POS   = {"Noun", "Verb", "Adjective"}
TERMINAL_POS = {"Verb", "Adjective"}

tokenized = []
for sent in sentences_raw:
    sent_clean = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", sent)
    tokens = [
        word for word, pos in okt.pos(sent_clean, norm=True, stem=True)
        if pos in TARGET_POS and len(word) > 1
    ]
    if tokens:
        tokenized.append(tokens)

print(f"[2] 형태소 분석 완료: {len(tokenized)}개 문장")

# ── 3. Word2Vec 학습 ─────────────────────────────────────────────
model = Word2Vec(
    sentences=tokenized,
    vector_size=120,
    window=5,
    min_count=2,
    workers=4,
    epochs=120,
    sg=1,
)
print(f"[3] Word2Vec 학습 완료 (어휘 수: {len(model.wv)})")

# ── 4. n-gram 전이 테이블 구축 ───────────────────────────────────
bigram  = defaultdict(list)
trigram = defaultdict(list)

for tokens in tokenized:
    for i in range(len(tokens) - 1):
        bigram[tokens[i]].append(tokens[i + 1])
    for i in range(len(tokens) - 2):
        trigram[(tokens[i], tokens[i + 1])].append(tokens[i + 2])

print(f"[4] n-gram 구축 완료 (bigram 키: {len(bigram)}, trigram 키: {len(trigram)})")

# ── 5. 종결 품사 판별 ────────────────────────────────────────────
def is_terminal(word):
    return any(pos in TERMINAL_POS for _, pos in okt.pos(word, norm=True, stem=True))

# ── 6. OOV 처리 ──────────────────────────────────────────────────
def edit_distance(a, b):
    """레벤슈타인 편집거리"""
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a):
        ndp = [i + 1]
        for j, cb in enumerate(b):
            ndp.append(min(dp[j] + (0 if ca == cb else 1),
                           dp[j + 1] + 1,
                           ndp[j] + 1))
        dp = ndp
    return dp[-1]

def resolve_oov(word):
    """
    OOV 단어를 어휘 내 단어로 변환.
    1단계: 형태소 분리 후 어휘에 있는 토큰 반환
    2단계: 편집거리 기반 가장 유사한 어휘 단어 반환
    """
    vocab = list(model.wv.index_to_key)

    # 1단계: 형태소 분리
    morphs = [
        tok for tok, pos in okt.pos(word, norm=True, stem=True)
        if pos in TARGET_POS and len(tok) > 1 and tok in model.wv
    ]
    if morphs:
        best = morphs[0]
        print(f"    [OOV] '{word}' → 형태소 분리 → '{best}' 로 시작")
        return best

    # 2단계: 편집거리 기반 유사 단어
    best = min(vocab, key=lambda v: (edit_distance(word, v), v))
    print(f"    [OOV] '{word}' → 편집거리 대체 → '{best}' 로 시작")
    return best

# ── 7. Temperature 샘플링 ────────────────────────────────────────
def softmax_sample(scores: dict, temperature: float) -> str:
    words  = list(scores.keys())
    logits = [scores[w] / max(temperature, 1e-8) for w in words]
    max_l  = max(logits)
    exps   = [math.exp(l - max_l) for l in logits]
    total  = sum(exps)
    probs  = [e / total for e in exps]
    return random.choices(words, weights=probs, k=1)[0]

# ── 8. 문장 생성 함수 ────────────────────────────────────────────
def generate_sentence(seed_word, min_len=4, max_len=15,
                      w2v_weight=0.4, top_k=5, temperature=1.0):
    # OOV 처리
    if seed_word not in model.wv:
        seed_word = resolve_oov(seed_word)

    sentence = [seed_word]

    while len(sentence) < max_len:
        current = sentence[-1]
        prev    = sentence[-2] if len(sentence) >= 2 else None

        # 후보 수집: trigram → bigram → w2v 순
        if prev and (prev, current) in trigram:
            candidates = list(set(trigram[(prev, current)]))
            freq_list  = trigram[(prev, current)]
        elif current in bigram:
            candidates = list(set(bigram[current]))
            freq_list  = bigram[current]
        else:
            candidates = [w for w, _ in model.wv.most_similar(current, topn=top_k)]
            freq_list  = candidates

        if not candidates:
            break

        # n-gram 빈도 점수
        freq_count = {c: freq_list.count(c) for c in candidates}
        total      = sum(freq_count.values()) or 1
        freq_score = {c: freq_count[c] / total for c in candidates}

        # w2v 유사도 점수
        w2v_score = {}
        for c in candidates:
            try:
                w2v_score[c] = model.wv.similarity(current, c) if c in model.wv else 0.0
            except:
                w2v_score[c] = 0.0

        # 최종 점수 + 반복 단어 패널티
        scores = {
            c: (1 - w2v_weight) * freq_score.get(c, 0)
               + w2v_weight      * w2v_score.get(c, 0)
            for c in candidates
        }
        for c in sentence:
            if c in scores:
                scores[c] *= 0.3

        next_word = softmax_sample(scores, temperature)
        sentence.append(next_word)

        # 동적 종결
        if len(sentence) >= min_len and is_terminal(next_word):
            break

    return " ".join(sentence)


In [ ]:
# ── 9. 문장 생성 예시 ────────────────────────────────────────────
# 정상 시드 + OOV 시드 혼합 테스트
seeds = [
    ("그리고",    "bool"),
]
import time

print("\n[5] 문장 생성 (OOV 처리 포함)")
print("=" * 60)
while True:
  for seed, tag in seeds:
      print(f"\n  ▶ 시드: [{seed}] ({tag})")
      for temp, label in [(0.5, "안정"), (1.0, "균형"), (1.8, "창의")]:
          sent = generate_sentence(seed, min_len=4, max_len=100,
                                  w2v_weight=0.4, temperature=temp)
          print(f"    temp={temp} ({label}) ({len(sent.split())}어절) {sent}")
  time.sleep(1)

In [ ]:
import re

# 파일에서 읽어오고 싶을 경우를 위해 함수화
def refine_corpus(text):
    # 1. 같은 메타데이터 태그 우선 제거 (선택 사항)
    #text = re.sub(r"\", "", text)

    # 2. 특수문자 제거 (한글, 영어, 숫자, 공백, 마침표만 남김)
    cleaned = re.sub(r"[^가-힣a-zA-Z0-9.\s]", "", text)

    # 3. 마침표 기준으로 나누되, 여러 개의 공백을 하나로 합치기
    # split('.') 대신 정규표현식을 쓰면 '!'나 '?' 기준 분리도 가능합니다.
    sentences = re.split(r"[.!?]", cleaned)

    # 4. 빈 문장 제거 및 양끝 공백 정리
    # 단어 개수가 너무 적은(예: 2단어 미만) 문장은 학습 가치가 낮으므로 필터링 추가
    sentences = [s.strip() for s in sentences if len(s.strip().split()) > 1]

    # 5. 중복 문장 제거 (방대한 글셋에서 혼란도를 낮추는 핵심 단계)
    sentences = list(dict.fromkeys(sentences))

    # 6. 줄바꿈 + / 삽입
    result = "\n/\n".join(sentences)
    return result

# 테스트
raw_text = input()
print(refine_corpus(raw_text))

In [ ]:
pip install sympy==1.12

In [ ]:
pip install --upgrade sympy

In [ ]:
"""
KoGPT2 파인튜닝 - 다중 파일 디렉토리 로딩
tsa_train/ 와 tsa_valid/ 내 모든 .txt 파일을 합산 학습
"""

import os, math, random, glob
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    GPT2LMHeadModel,
    PreTrainedTokenizerFast,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

# ══════════════════════════════════════════════════════════════════
# 설정
# ══════════════════════════════════════════════════════════════════
CFG = {
    "model_name"  : "skt/kogpt2-base-v2",
    "train_dir"   : "./tsa_train",   # 학습 txt 파일들이 있는 디렉토리
    "valid_dir"   : "./tsa_valid",   # 검증 txt 파일들이 있는 디렉토리
    "output_dir"  : "./kogpt2_finetuned",
    "max_len"     : 256,
    "batch_size"  : 8,
    "grad_accum"  : 4,
    "epochs"      : 1,
    "lr"          : 3e-5,
    "warmup_ratio": 0.1,
    "save_every"  : 1,
    "patience"    : 3,              # early stopping
    "seed"        : 42
}

random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {device}")

# ══════════════════════════════════════════════════════════════════
# 디렉토리 내 모든 txt 파일 로딩
# ══════════════════════════════════════════════════════════════════
def load_dir(directory: str) -> list[str]:
    """디렉토리 내 모든 .txt 파일을 읽어 라인 리스트로 반환"""
    files = sorted(glob.glob(os.path.join(directory, "*.txt")))
    if not files:
        raise FileNotFoundError(f"'{directory}' 에 .txt 파일이 없습니다.")

    all_lines = []
    for fpath in files:
        with open(fpath, "r", encoding="utf-8") as f:
            lines = [l.strip() for l in f if l.strip()]
        all_lines.extend(lines)
        print(f"  └ {os.path.basename(fpath)}: {len(lines):,}개")

    print(f"  → 합계: {len(all_lines):,}개\n")
    return all_lines

# ══════════════════════════════════════════════════════════════════
# 토크나이저 & 모델
# ══════════════════════════════════════════════════════════════════
print("[1] 모델 로드 중...")
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    CFG["model_name"],
    bos_token="</s>", eos_token="</s>",
    unk_token="<unk>", pad_token="<pad>",
    mask_token="<mask>",
)
model = GPT2LMHeadModel.from_pretrained(CFG["model_name"])
model.resize_token_embeddings(len(tokenizer))
model.to(device)
print(f"   파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

# ══════════════════════════════════════════════════════════════════
# 데이터셋
# ══════════════════════════════════════════════════════════════════
class MultiFileDataset(Dataset):
    def __init__(self, lines: list[str], tokenizer, max_len: int):
        self.examples = []
        for line in lines:
            enc = tokenizer(
                line,
                max_length=max_len,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
            input_ids      = enc["input_ids"].squeeze()
            attention_mask = enc["attention_mask"].squeeze()
            labels         = input_ids.clone()
            labels[attention_mask == 0] = -100
            self.examples.append((input_ids, attention_mask, labels))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


print("[2] 데이터 로드 중...")
print(f"  [train_dir: {CFG['train_dir']}]")
train_lines = load_dir(CFG["train_dir"])

print(f"  [valid_dir: {CFG['valid_dir']}]")
valid_lines = load_dir(CFG["valid_dir"])

train_ds = MultiFileDataset(train_lines, tokenizer, CFG["max_len"])
valid_ds = MultiFileDataset(valid_lines, tokenizer, CFG["max_len"])
print(f"  최종 → train: {len(train_ds):,}개 / valid: {len(valid_ds):,}개")

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

# ══════════════════════════════════════════════════════════════════
# 옵티마이저 & 스케줄러
# ══════════════════════════════════════════════════════════════════
total_steps  = (len(train_loader) // CFG["grad_accum"]) * CFG["epochs"]
warmup_steps = int(total_steps * CFG["warmup_ratio"])

optimizer = AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# ══════════════════════════════════════════════════════════════════
# 학습 루프 (early stopping 포함)
# ══════════════════════════════════════════════════════════════════
os.makedirs(CFG["output_dir"], exist_ok=True)
best_valid_loss = float("inf")
patience_count  = 0

print(f"\n[3] 학습 시작 (총 {CFG['epochs']}epoch / {total_steps}스텝)")
print("=" * 60)

for epoch in range(1, CFG["epochs"] + 1):

    # ── train ──────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    for step, (input_ids, attention_mask, labels) in enumerate(train_loader, 1):
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)

        out  = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = out.loss / CFG["grad_accum"]
        loss.backward()
        train_loss += out.loss.item()

        if step % CFG["grad_accum"] == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        if step % 50 == 0:
            print(f"  epoch {epoch} | step {step}/{len(train_loader)} | loss {train_loss/step:.4f}")

    avg_train = train_loss / len(train_loader)

    # ── valid ───────────────────────────────────────────────────
    model.eval()
    valid_loss = 0.0
    with torch.no_grad():
        for input_ids, attention_mask, labels in valid_loader:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels         = labels.to(device)
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            valid_loss += out.loss.item()

    avg_valid = valid_loss / len(valid_loader)
    ppl       = math.exp(avg_valid)
    print(f"\n  ▶ epoch {epoch} | train={avg_train:.4f} | valid={avg_valid:.4f} | PPL={ppl:.2f}")

    # ── 저장 ────────────────────────────────────────────────────
    if epoch % CFG["save_every"] == 0:
        path = os.path.join(CFG["output_dir"], f"epoch{epoch}")
        model.save_pretrained(path)
        tokenizer.save_pretrained(path)
        print(f"  → 저장: {path}")

    # ── early stopping ──────────────────────────────────────────
    if avg_valid < best_valid_loss:
        best_valid_loss = avg_valid
        patience_count  = 0
        model.save_pretrained(os.path.join(CFG["output_dir"], "best"))
        tokenizer.save_pretrained(os.path.join(CFG["output_dir"], "best"))
        print(f"  → best 모델 갱신 (valid_loss={best_valid_loss:.4f})")
    else:
        patience_count += 1
        print(f"  ⚠ 개선 없음 ({patience_count}/{CFG['patience']})")
        if patience_count >= CFG["patience"]:
            print(f"  Early stopping! (epoch {epoch})")
            break

    print("-" * 60)

print(f"\n[4] 학습 완료! best valid_loss={best_valid_loss:.4f}")

# ══════════════════════════════════════════════════════════════════
# 추론
# ══════════════════════════════════════════════════════════════════
def chat(prompt: str, max_new_tokens: int = 128, temperature: float = 0.8, top_p: float = 0.9):
    model.eval()
    # 두 포맷 모두 지원
    text = f"<s> 다음 내용을 설명하라: {prompt}"
    input_ids = tokenizer.encode(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.3,
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    if "설명하라:" in decoded:
        decoded = decoded.split("설명하라:")[-1].strip()
    elif "답변:" in decoded:
        decoded = decoded.split("답변:")[-1].strip()
    return decoded


print("\n[5] 샘플 추론")
print("=" * 60)
for q in [
    "한국어의 교착어적 특성은 무엇인가요?",
    "알타이 제어의 특징은 무엇인가요?",
    "인공지능 윤리 문제란 무엇인가요?",
]:
    print(f"Q: {q}\nA: {chat(q)}\n")

In [ ]:
print("\n[5] 샘플 추론")
print("=" * 60)
for q in [
    "안녕",
]:
    print(f"Q: {q}\nA: {chat(q)}\n")

In [ ]:
# 인터럽트 후 model, tokenizer 객체는 메모리에 살아있음
import os
os.makedirs("./kogpt2_emergency_save", exist_ok=True)
model.save_pretrained("./kogpt2_emergency_save")
tokenizer.save_pretrained("./kogpt2_emergency_save")
print("저장 완료")

In [ ]:
import torch
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast

# ── 모델 로드 ──────────────────────────────────────────────────
MODEL_PATH = "./kogpt2_emergency_save"  # 저장 경로 맞게 수정

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    MODEL_PATH,
    bos_token="</s>", eos_token="</s>",
    unk_token="<unk>", pad_token="<pad>",
    mask_token="<mask>",
)
model = GPT2LMHeadModel.from_pretrained(MODEL_PATH).to(device)
model.eval()
print(f"모델 로드 완료 ({device})")

# ── 추론 함수 ──────────────────────────────────────────────────
def chat(prompt, max_new_tokens=150, temperature=0.8, top_p=0.9):
    text = f"<s> 다음 내용을 설명하라: {prompt}"
    input_ids = tokenizer.encode(text, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.3,
        )

    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    if "설명하라:" in decoded:
        decoded = decoded.split("설명하라:")[-1].strip()
    return decoded

# ── 테스트 ─────────────────────────────────────────────────────
while True:
    q = input("\n질문 입력 (종료: q): ").strip()
    if q == "q":
        break
    print(f"A: {chat(q)}")

In [ ]:
import torch, random, re

# ── 프롬프트 시드 풀 ───────────────────────────────────────────
SEED_POOL = [
    "인공지능", "경제 성장", "주식 시장", "기술 발전", "사회 변화",
    "환경 문제", "의료 기술", "교육 시스템", "정치 구조", "노동 시장",
    "반도체", "금리 인상", "탄소중립", "빅데이터", "플랫폼 경제",
]

# ── 생성 함수 ──────────────────────────────────────────────────
def generate(prompt, max_new_tokens=100, temperature=0.85, top_p=0.92):
    text = f"<s> 다음 내용을 설명하라: {prompt}"
    ids  = tokenizer.encode(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.3,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    if "설명하라:" in decoded:
        decoded = decoded.split("설명하라:")[-1].strip()
    return decoded

# ── 응답에서 다음 프롬프트 자동 추출 ──────────────────────────
def extract_next_prompt(text: str) -> str:
    """
    생성된 문장에서 명사구를 뽑아 다음 시드로 사용.
    1순위: 2~6글자 명사처럼 보이는 단어
    2순위: 랜덤 시드
    """
    # 한글 단어 추출 (2~6글자)
    words = re.findall(r'[가-힣]{2,6}', text)
    # 불용어 제거
    stopwords = {"있다","없다","하다","된다","이다","한다","것이","때문","위해","통해","대한","관련","경우","부분","가장","이번","지난","현재","이후","이전","또한","따라","대해"}
    candidates = [w for w in words if w not in stopwords]
    if candidates:
        # 뒤쪽 단어일수록 새로운 주제일 가능성 높음
        return random.choice(candidates[-5:]) if len(candidates) >= 5 else random.choice(candidates)
    return random.choice(SEED_POOL)

# ── 자가 생성 루프 ─────────────────────────────────────────────
def self_loop(start_prompt=None, rounds=20, delay=0.5):
    import time
    prompt = start_prompt or random.choice(SEED_POOL)
    history = []

    print("=" * 65)
    print(f"  자가 생성 루프 시작 / {rounds}라운드")
    print("=" * 65)

    for i in range(1, rounds + 1):
        response = generate(prompt)
        next_prompt = extract_next_prompt(response)

        history.append({"round": i, "prompt": prompt, "response": response, "next": next_prompt})

        print(f"\n[{i:02d}] 프롬프트 : {prompt}")
        print(f"      응답     : {response[:120]}{'...' if len(response)>120 else ''}")
        print(f"      다음시드  : {next_prompt}")
        print("-" * 65)

        prompt = next_prompt
        time.sleep(delay)

    return history

# ── 실행 ───────────────────────────────────────────────────────
history = self_loop(start_prompt="인공지능", rounds=20)

In [ ]:
"""
KoGPT2 파인튜닝 - 질문/답변 쌍 토큰 기반 학습 + 토큰 확률 최적 추론
══════════════════════════════════════════════════════════════════════

[데이터 형식]  tsa_train/*.txt, tsa_valid/*.txt
    한 줄 = 질문\t답변
    예) 알타이 제어의 특징은?\t알타이 제어는 교착어 특성을 지니며...

[학습]
    프롬프트 : <s> 질문: {q} 답변: {a} </s>
    레이블   : 질문 토큰 영역 → -100 마스킹 (답변 토큰만 loss 계산)

[추론 파이프라인]
    Step 1. Diverse Beam Search  → num_beams 개 후보
    Step 2. Top-k/Top-p Sampling → n_sample 개 후보
    Step 3. 모든 후보를 token log-prob 평균으로 재랭킹
    Step 4. Length Penalty 적용 후 최적 답변 1개 반환
"""

import os, math, random, glob
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    GPT2LMHeadModel,
    PreTrainedTokenizerFast,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW

# ══════════════════════════════════════════════════════════════════
# 설정
# ══════════════════════════════════════════════════════════════════
CFG = {
    # 모델 / 경로
    "model_name"    : "skt/kogpt2-base-v2",
    "train_dir"     : "./tsa_train",
    "valid_dir"     : "./tsa_valid",
    "output_dir"    : "./kogpt2_finetuned",
    # 학습
    "max_len"       : 256,
    "batch_size"    : 8,
    "grad_accum"    : 4,
    "epochs"        : 5,
    "lr"            : 3e-5,
    "warmup_ratio"  : 0.1,
    "save_every"    : 1,
    "patience"      : 3,
    "seed"          : 42,
    # 추론 - Beam
    "num_beams"         : 6,       # Diverse Beam Search 후보 수
    "num_beam_groups"   : 3,       # 빔 그룹 수 (다양성 확보)
    "diversity_penalty" : 1.0,     # 그룹 간 다양성 패널티
    "no_repeat_ngram"   : 3,       # n-gram 반복 억제
    # 추론 - Sampling
    "n_sample"      : 4,           # 샘플링 후보 수
    "temperature"   : 0.8,
    "top_k"         : 50,
    "top_p"         : 0.92,
    # 추론 - 공통
    "max_new_tokens"    : 128,
    "rep_penalty"       : 1.3,
    "length_penalty_alpha": 0.6,   # 길이 보정 지수 (0=없음, 1=비례)
    "min_ans_tokens"    : 5,       # 유효 답변 최소 토큰 수
}

PROMPT_Q  = "질문: "
PROMPT_A  = " 답변: "

random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] {device}")


# ══════════════════════════════════════════════════════════════════
# 데이터 로딩
# ══════════════════════════════════════════════════════════════════
def load_qa_dir(directory: str) -> list[tuple[str, str]]:
    """디렉토리 내 모든 .txt 파일에서 '질문\t답변' 쌍 로드"""
    files = sorted(glob.glob(os.path.join(directory, "*.txt")))
    if not files:
        raise FileNotFoundError(f"'{directory}' 에 .txt 파일이 없습니다.")

    pairs: list[tuple[str, str]] = []
    for fpath in files:
        file_pairs = []
        with open(fpath, "r", encoding="utf-8") as f:
            for lineno, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                if "\t" not in line:
                    print(f"  [경고] {os.path.basename(fpath)} "
                          f"L{lineno}: 탭 없음, 건너뜀 → {line[:40]}")
                    continue
                q, a = line.split("\t", maxsplit=1)
                if q.strip() and a.strip():
                    file_pairs.append((q.strip(), a.strip()))
        pairs.extend(file_pairs)
        print(f"  └ {os.path.basename(fpath)}: {len(file_pairs):,}쌍")

    print(f"  → 합계: {len(pairs):,}쌍\n")
    return pairs


# ══════════════════════════════════════════════════════════════════
# 데이터셋 - 답변 토큰만 loss 계산
# ══════════════════════════════════════════════════════════════════
class QADataset(Dataset):
    """
    입력  : <s> 질문: {q} 답변: {a} </s>
    레이블 : 질문 구간 → -100, 답변 구간 → 토큰 ID
    """

    def __init__(self, pairs, tokenizer, max_len):
        self.examples = []
        bos = tokenizer.bos_token or ""
        eos = tokenizer.eos_token or ""

        for q, a in pairs:
            full_text = f"{bos}{PROMPT_Q}{q}{PROMPT_A}{a}{eos}"
            enc = tokenizer(
                full_text,
                max_length=max_len,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
            input_ids      = enc["input_ids"].squeeze()
            attention_mask = enc["attention_mask"].squeeze()

            # 질문 프롬프트 토큰 길이 (마스킹 기준)
            q_prompt  = f"{bos}{PROMPT_Q}{q}{PROMPT_A}"
            q_tok_len = len(tokenizer(q_prompt, add_special_tokens=False)["input_ids"])

            labels = input_ids.clone()
            labels[:q_tok_len]          = -100   # 질문 구간 마스킹
            labels[attention_mask == 0] = -100   # 패딩 마스킹

            self.examples.append((input_ids, attention_mask, labels))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


# ══════════════════════════════════════════════════════════════════
# 모델 / 토크나이저
# ══════════════════════════════════════════════════════════════════
print("[1] 모델 로드...")
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    CFG["model_name"],
    bos_token="</s>", eos_token="</s>",
    unk_token="<unk>", pad_token="<pad>",
    mask_token="<mask>",
)
model = GPT2LMHeadModel.from_pretrained(CFG["model_name"])
model.resize_token_embeddings(len(tokenizer))
model.to(device)
print(f"   파라미터: {sum(p.numel() for p in model.parameters()):,}")


# ══════════════════════════════════════════════════════════════════
# 데이터 준비
# ══════════════════════════════════════════════════════════════════
print("\n[2] 데이터 로드...")
print(f"  [train: {CFG['train_dir']}]")
train_pairs = load_qa_dir(CFG["train_dir"])
print(f"  [valid: {CFG['valid_dir']}]")
valid_pairs = load_qa_dir(CFG["valid_dir"])

train_ds = QADataset(train_pairs, tokenizer, CFG["max_len"])
valid_ds = QADataset(valid_pairs, tokenizer, CFG["max_len"])
print(f"  → train {len(train_ds):,} / valid {len(valid_ds):,}")

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=True)


# ══════════════════════════════════════════════════════════════════
# 옵티마이저 / 스케줄러
# ══════════════════════════════════════════════════════════════════
total_steps  = (len(train_loader) // CFG["grad_accum"]) * CFG["epochs"]
warmup_steps = int(total_steps * CFG["warmup_ratio"])

optimizer = AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)


# ══════════════════════════════════════════════════════════════════
# 학습 루프
# ══════════════════════════════════════════════════════════════════
os.makedirs(CFG["output_dir"], exist_ok=True)
best_valid_loss = float("inf")
patience_count  = 0

print(f"\n[3] 학습 시작 ({CFG['epochs']}epoch / {total_steps}step)")
print("=" * 60)

for epoch in range(1, CFG["epochs"] + 1):

    # ── Train ─────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    for step, (input_ids, attention_mask, labels) in enumerate(train_loader, 1):
        input_ids, attention_mask, labels = (
            input_ids.to(device), attention_mask.to(device), labels.to(device)
        )
        out  = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = out.loss / CFG["grad_accum"]
        loss.backward()
        train_loss += out.loss.item()

        if step % CFG["grad_accum"] == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()

        if step % 50 == 0:
            print(f"  epoch {epoch} | step {step}/{len(train_loader)} "
                  f"| loss {train_loss/step:.4f}")

    avg_train = train_loss / len(train_loader)

    # ── Valid ─────────────────────────────────────────────────────
    model.eval()
    valid_loss = 0.0
    with torch.no_grad():
        for input_ids, attention_mask, labels in valid_loader:
            input_ids, attention_mask, labels = (
                input_ids.to(device), attention_mask.to(device), labels.to(device)
            )
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            valid_loss += out.loss.item()

    avg_valid = valid_loss / len(valid_loader)
    ppl       = math.exp(avg_valid)
    print(f"\n  ▶ epoch {epoch} | train={avg_train:.4f} | "
          f"valid={avg_valid:.4f} | PPL={ppl:.2f}")

    # ── 저장 ─────────────────────────────────────────────────────
    if epoch % CFG["save_every"] == 0:
        save_path = os.path.join(CFG["output_dir"], f"epoch{epoch}")
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"  → 저장: {save_path}")

    # ── Early Stopping ────────────────────────────────────────────
    if avg_valid < best_valid_loss:
        best_valid_loss = avg_valid
        patience_count  = 0
        model.save_pretrained(os.path.join(CFG["output_dir"], "best"))
        tokenizer.save_pretrained(os.path.join(CFG["output_dir"], "best"))
        print(f"  → best 갱신 (valid_loss={best_valid_loss:.4f})")
    else:
        patience_count += 1
        print(f"  ⚠ 개선 없음 ({patience_count}/{CFG['patience']})")
        if patience_count >= CFG["patience"]:
            print(f"  Early Stopping (epoch {epoch})")
            break

    print("-" * 60)

print(f"\n[4] 학습 완료! best valid_loss={best_valid_loss:.4f}")


# ══════════════════════════════════════════════════════════════════
# 추론 엔진
# ══════════════════════════════════════════════════════════════════

def _make_prompt(question: str) -> str:
    """추론용 프롬프트 (답변 이전까지)"""
    bos = tokenizer.bos_token or ""
    return f"{bos}{PROMPT_Q}{question}{PROMPT_A}"


def _extract_answer(decoded: str) -> str:
    """디코딩 결과에서 답변 부분만 추출"""
    marker = PROMPT_A.strip()   # "답변:"
    if marker in decoded:
        return decoded.split(marker)[-1].strip()
    return decoded.strip()


@torch.no_grad()
def _token_score(question: str, answer: str) -> float:
    """
    토큰 log-prob 기반 답변 신뢰도 점수 (Length Penalty 적용).

    score = mean(log P(token_i)) / (answer_len ^ alpha)

    alpha=0 이면 길이 보정 없음, alpha>0 이면 긴 답변 소폭 우대.
    """
    model.eval()
    eos  = tokenizer.eos_token or ""
    full = _make_prompt(question) + answer + eos
    ids  = tokenizer(full, return_tensors="pt")["input_ids"].to(device)

    # 프롬프트 토큰 길이
    q_len = len(tokenizer(_make_prompt(question),
                          add_special_tokens=False)["input_ids"])

    logits    = model(input_ids=ids).logits[0]          # (seq, vocab)
    log_probs = F.log_softmax(logits[:-1], dim=-1)      # shift
    token_lps = log_probs[range(len(ids[0]) - 1), ids[0, 1:]]

    ans_lps = token_lps[q_len - 1:]                     # 답변 구간
    n       = len(ans_lps)
    if n < CFG["min_ans_tokens"]:                        # 너무 짧으면 패널티
        return float("-inf")

    mean_lp = ans_lps.mean().item()
    # Length Penalty: 긴 답변을 소폭 우대 (alpha > 0)
    lp_score = mean_lp / (n ** CFG["length_penalty_alpha"])
    return lp_score


@torch.no_grad()
def _generate_diverse_beam(question: str) -> list[str]:
    """Diverse Beam Search → 여러 후보 답변"""
    prompt_ids = tokenizer.encode(_make_prompt(question),
                                  return_tensors="pt").to(device)
    outputs = model.generate(
        prompt_ids,
        max_new_tokens=CFG["max_new_tokens"],
        num_beams=CFG["num_beams"],
        num_beam_groups=CFG["num_beam_groups"],
        diversity_penalty=CFG["diversity_penalty"],
        num_return_sequences=CFG["num_beams"],
        no_repeat_ngram_size=CFG["no_repeat_ngram"],
        repetition_penalty=CFG["rep_penalty"],
        early_stopping=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    candidates = []
    for seq in outputs:
        decoded = tokenizer.decode(seq, skip_special_tokens=True)
        candidates.append(_extract_answer(decoded))
    return candidates


@torch.no_grad()
def _generate_sampling(question: str) -> list[str]:
    """Top-k + Top-p Sampling → 다양한 후보 답변"""
    prompt_ids = tokenizer.encode(_make_prompt(question),
                                  return_tensors="pt").to(device)
    outputs = model.generate(
        prompt_ids,
        max_new_tokens=CFG["max_new_tokens"],
        do_sample=True,
        temperature=CFG["temperature"],
        top_k=CFG["top_k"],
        top_p=CFG["top_p"],
        num_return_sequences=CFG["n_sample"],
        repetition_penalty=CFG["rep_penalty"],
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    candidates = []
    for seq in outputs:
        decoded = tokenizer.decode(seq, skip_special_tokens=True)
        candidates.append(_extract_answer(decoded))
    return candidates


def rank_candidates(
    question: str,
    candidates: list[str],
) -> list[tuple[str, float]]:
    """후보 답변 전체를 token log-prob 점수로 정렬"""
    scored = [(ans, _token_score(question, ans)) for ans in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored


def chat(question: str, verbose: bool = True) -> str:
    """
    통합 추론 함수
    ─────────────────────────────────────────────
    1) Diverse Beam Search 후보 (num_beams개)
    2) Top-k/Top-p Sampling 후보 (n_sample개)
    3) 전체 후보 token log-prob 재랭킹
    4) Length Penalty 보정 후 최적 답변 반환
    """
    model.eval()

    # ── 후보 생성 ──────────────────────────────────────────────
    beam_cands    = _generate_diverse_beam(question)
    sample_cands  = _generate_sampling(question)
    all_cands     = list(dict.fromkeys(beam_cands + sample_cands))  # 중복 제거

    # ── 토큰 확률 재랭킹 ──────────────────────────────────────
    ranked = rank_candidates(question, all_cands)

    if verbose:
        print(f"\n{'='*64}")
        print(f"Q: {question}")
        print(f"{'─'*64}")
        print(f"[후보 {len(ranked)}개 / 토큰 log-prob 재랭킹 결과]")
        for i, (ans, score) in enumerate(ranked, 1):
            tag = " ◀ 최적" if i == 1 else ""
            src = "Beam" if ans in beam_cands else "Sample"
            print(f"  {i:2d}위 [{src}] score={score:+.4f}{tag}")
            print(f"      {ans[:80]}{'...' if len(ans)>80 else ''}")
        print()

    best_answer = ranked[0][0] if ranked else "답변을 생성할 수 없습니다."

    if verbose:
        print(f"A: {best_answer}\n")

    return best_answer


# ══════════════════════════════════════════════════════════════════
# 샘플 추론
# ══════════════════════════════════════════════════════════════════
print("\n[5] 샘플 추론")

test_questions = [
    "한국어의 교착어적 특성은 무엇인가요?",
    "알타이 제어의 특징은 무엇인가요?",
    "인공지능 윤리 문제란 무엇인가요?",
]

for q in test_questions:
    chat(q, verbose=True)